# Row Level Security (RLS) — `vstone_catalog.security`

**Group -> Brand access matrix:**

| Group | Brands visible |
|---|---|
| `admin_group` | All brands (unrestricted) |
| `premium_users` | `bmw`, `mercedes-benz`, `lexus` |
| `toyota_users` | `toyota` only |
| `honda_users` | `honda` only |
| *(others)* | No rows (deny by default) |


## Step 1 -- Inspect the base Gold table

In [0]:
-- ============================================================
-- STEP 1: Inspect the base Gold table we will secure
-- ============================================================
SELECT * FROM vstone_catalog.gold.agg_top_10_brands_by_spend
ORDER BY total_market_value_usd DESC;

## Step 2 -- Verify workspace groups

In [0]:
-- ============================================================
-- STEP 2: Verify your workspace groups exist
-- ============================================================
SHOW GROUPS;

## Step 3 -- Confirm user identity and membership

In [0]:
-- ============================================================
-- STEP 3: Confirm current user identity and group membership
-- ============================================================
SELECT
  CURRENT_USER()                         AS current_user,
  IS_MEMBER('toyota_users')              AS is_toyota_user,
  IS_MEMBER('honda_users')               AS is_honda_user,
  IS_MEMBER('premium_users')             AS is_premium_user,
  is_account_group_member('admins') AS is_admin;

## Step 4 -- Create RLS view in `security` schema

In [0]:
-- ============================================================
-- STEP 4: Create the RLS governance view
-- Location: vstone_catalog.security.rls_brand_market_data
-- ============================================================
CREATE OR REPLACE VIEW vstone_catalog.security.rls_brand_market_data AS
SELECT
  brand,
  total_market_value_usd,
  total_listings,
  avg_price_usd,
  gold_load_dt
FROM vstone_catalog.gold.agg_top_10_brands_by_spend
WHERE
  -- CASE evaluated per-row per-user; IS_MEMBER() resolves at query time
  CASE
    -- Admin sees everything
    WHEN is_account_group_member('admins')
      THEN TRUE

    -- Premium segment: BMW, Mercedes-Benz, Lexus
    WHEN IS_MEMBER('premium_users')
      THEN brand IN ('bmw', 'mercedes-benz', 'lexus')

    -- Brand-specific groups
    WHEN IS_MEMBER('toyota_users')
      THEN brand = 'toyota'

    WHEN IS_MEMBER('honda_users')
      THEN brand = 'honda'

    -- Deny all unrecognised users
    ELSE FALSE
  END;

## Step 5 -- Verify filtered output

In [0]:
-- ============================================================
-- STEP 5: Verify the RLS view returns correctly filtered rows
-- ============================================================
SELECT * FROM vstone_catalog.security.rls_brand_market_data
ORDER BY total_market_value_usd DESC;

## Step 6 -- Lock down the base table

In [0]:
-- ============================================================
-- STEP 6: Apply access control on the base Gold table
-- Revoke direct SELECT on the base table for non-admin users.
-- This forces all reads to go through the security view.
--
-- Run as a workspace admin or table owner:
-- ============================================================

-- Grant governed security view to all workspace users
GRANT SELECT ON VIEW vstone_catalog.security.rls_brand_market_data
  TO `account users`;                          

-- Revoke direct Gold table access
REVOKE SELECT ON TABLE vstone_catalog.gold.agg_top_10_brands_by_spend
  FROM `account users`;

-- Keep admins on base tables for ETL/debugging
GRANT SELECT ON TABLE vstone_catalog.gold.agg_top_10_brands_by_spend
  TO `akashmishraa202@gmail.com`;                        

## Step 7 -- RLS on `fact_listings` (transaction grain)

In [0]:
-- ============================================================
-- STEP 7: Also apply RLS on fact_listings for row-level brand
-- ============================================================
CREATE OR REPLACE VIEW vstone_catalog.security.rls_fact_listings AS
SELECT
  listing_id,
  brand,
  model,
  manufacture_year,
  listing_date,
  price_rub,
  price_usd,
  price_category,
  mileage_km,
  fuel_type,
  transmission_type,
  location_key,
  photo_count,
  car_age_at_listing,
  is_high_mileage,
  price_per_hp_usd,
  gold_load_dt
FROM vstone_catalog.gold.fact_listings
WHERE
  CASE
    WHEN is_account_group_member('admin_group') THEN TRUE
    WHEN IS_MEMBER('premium_users')  THEN brand IN ('bmw', 'mercedes-benz', 'lexus')
    WHEN IS_MEMBER('toyota_users')   THEN brand = 'toyota'
    WHEN IS_MEMBER('honda_users')    THEN brand = 'honda'
    ELSE FALSE
  END;

-- Verify
SELECT brand, COUNT(*) AS row_count
FROM vstone_catalog.security.rls_fact_listings
GROUP BY brand
ORDER BY row_count DESC;